# Proofreading spine detection - accuracy of mesh reconstruction

With this notebook you help us to assess the quality of the spine detection and segmentation algorithm.
To ensure the accuracy of the results,

 **please read all instructions in the notebook carefully** and follow them!

 **Please do not use "run all cells" in this notebook. Instead read the instructions for each cell and then run it individually, one by one.**

 ## Your assigned part of a neuron

 If you read this, you should have been assigned by the administrators a neuron, identified by its `entity id` and one or several sections of that neuron, identified by their `section id`. If you do not know your assignments, then please do not continue and contact one of the project admins.

 ## Basic mode of working
 The main part of the notebook will present you a 3d rendering of a detected and reconstructed spine with parts of the source neuron mesh for context.

 In the 3d rendering you will see the following parts:
   - The detected `neck` of a spine in green
   - The detected `head` of a spine in red. If the spine has no clearly separable head, this part can be missing
   - Points on the surface of the source neuron mesh in semi-transparent blue

  See below an example

![image](./example1.png)

It is your task to characterize the potential errors that may have occured in the reconstruction of the spine mesh and in the separation of spine head from spine neck.

## Potential errors
In this notebook, we are interested six types of errors in two classes. To explain the types of error, we define a *detected spine* as a spine that has been detected by our algorithm and is displayed to you. A *true spine* is a spine detected in the surface mesh (blue points) by the expert, i.e., by you.

  - Mesh reconstruction errors:
    - **Spine mergers**: More than one true spine is part of a detected spine
    - **Spine split**: A detected spine contains less than 90% percent of a true spine
  - Head/neck separation errors:
    - **Separation shifted**: When the separation between head and neck is broadly in the right location, but shifted; i.e., too much neck or too much head
    - **Separation broken**: When the separation between head and neck is completely nonsensical.
    - **False head**: When a head has been annotated although the spine does not have a separable head
    - **Missing head**: When no head has been annotated although the spine does have a separable head.

All assessments above will be based - in part - on opinion. Spines come in all kinds of shapes and it can be legitimately difficult and ambiguous to draw the separation between head and neck. It is not possible to build an algorithm that matches the opinion of all neuroscientists. Therefore, we define

### Rule 1:
For all assessments ask yourself: Is it conceivable that a reasonable human reconstructor would accept the presented solution? If so, do not annotate it as an error.

## Examples:

### Spine merger
![image](./merger_example.png)

We see that two spines are getting close and "kiss" each other. This resulted in the algorithm detecting them as a single spine - a spine merger. This also messed up the head / neck detection.

This type of error is ususally very obvious and easy to detect.

However, remember 
#### Rule 2
It is not a spine merger if both parts emerge from a common base (see below for an example).

### Spine split
![image](./spine_split.png)

We see that a short stubby spine was detected, but the reconstructed mesh is split in half and covers only part of it.

Please only annotate sufficiently severe cases where a spine looks like it is "split on half". Some missing mesh around the base of the spine is not sufficient.

### Separation shifted
![image](./separation1.png)

We see clearly that parts of the head of the spine are rendered in green, hence detected as neck. The overall separation between head and neck is almost proper, but shifted in favor of the neck.

When evaluating this type of error, keep Rule #1 in mind. Also make sure to view the spine **from all possible angles**.

### Separation broken
![image](./separation_broken.png)

We see a very nonsensical head/neck separation. Not only are large parts of the neck annotated as head, but there is another patch of neck within the head. Completely broken.

This type of error is usually very obvious and easy to detect.

### False head

![image](./false_head.png)

We see a very stubby spine that is almost completely annotated as head. But there is no separate head and in that case, a spine is supposed to be all neck.

**This type of error is tricky to evaluate.** Often, from one camera angle no separate head can be seen, but when shifting the camera a separate head becomes clearly visible.

#### Rule 3
View a spine from multiple angles before tagging an error.

### Missing head
![image](./missing_head.png)

We see rather stubby spine where large parts of the neck are missing. The remaining part has been annotated as neck although it may also be categorized as a head. This type of error is very rare and happens almost exclusively in conjunction with other errors. The example above is frankly not very good, but the case is so rare that a better one could not be readily found.


## Error combinations

Some types of errors can appear together. Above, we have already seen some examples.

The example for "spine merger" combines that error with "separation broken". The example for "missing head" combines that error with "spine split".

Here is an example of three errors together.

![image](./three_errors.png)

We see a combination of spine merger, split spine (the right one) and missing head (also the right one).


### Not errors

![image](./no_error1.png)

At first glance, it appears like two spines have been merged. But upon closer inspection we see that they originate from the same base. Hence, this is correctly reconstructed as a single, split spine.

One can argue that the head/neck separation of the right spine is a bit shifted, but I think it is still OK.

![image](./no_error2.png)

We see that the reconstructed mesh for the spine includes parts of the dendrite surface. This is ugly, but in this notebook this is **not an error we are concerned about**. This type of error is evaluated in a different notebook.

## Controls

To report errors, the following controls are available: (This is a screenshot; do not try to click them)

![image](./controls.png)

In general, you use the bottom parts of the control to indicate the types of errors found, then use the button in the first row to move to the next spine.


### First row:
From left to right, we see:

A label detailing how many more detected spines you are expected to evaluate.

A button that stores your evaluation for the currently selected spine and moves to the next. When you are done with all spines, this button disappears.

A button that you can use to take a screenshot (to be later displayed below). **This is optional**. If you encounter interesting spines or weird errors you can document them with a screenshot and we may use it in the article. Again: optional.

### Second row:
The second row is a check box that you can use to indicate that the currently displayed spine is not really a spine - a false positive. This disables all other types of errors - without a real spine there are no real errors.

### Rows three to five:
With these rows you can report mesh reconstruction errors. Report spine mergers by indicating the *number of true spines* that are in any way part of the detected spine. Report the presence of a spine split with the check box.

### Rows six and seven:
With these rows you can report head/neck separation errors. Check all types of error that apply.

## Evaluating all spines
Each time you hit "Save and next" you will move on to the next spine. Continue this until you have evaluated all spines *and the button disappears*. 

After that **continue with the remaining cells of the notebook!**

# Moving on...

Having read all of the above, you are in a great position to start proofreading and error assessment.

Execute the following cells one by one to download the data and launch the 3d view.

## Imports and select project

Select the project you want to work with. 
Important: Select the spine proofreading project that you have been invited to by the administrators.

In [ ]:
from morph_spines import load_morphology_with_spines

import obi_auth
import numpy
import pandas

from entitysdk import Client, types, models
from obi_one import CellMorphologyFromID
from entitysdk.models import EMCellMesh, EMDenseReconstructionDataset, CellMorphology
from obi_notebook. get_projects import get_projects
from obi_notebook.get_environment import get_environment
from ipywidgets import widgets
from IPython import display
from scipy.spatial import KDTree

import logging
loggers = [logging.getLogger(name) for name in logging.root.manager.loggerDict]
for logger in loggers:
    logger.setLevel(logging.ERROR)

env_ = get_environment()
token = obi_auth.get_token(environment=env_, auth_mode="daf")
project_context = get_projects(token, env=env_)

## Accessing and downloading the data

**IMPORTANT. Read this carefully** 

You have been assigned an `entity id` of the neuron to proofread. Please fill it in where "PASTE ENTITY ID IN HERE" is written.

You have also been assigned a list of `section ids` to proofread. Please fill in the section ids *as a list* where "PASTE SECTION IDS HERE" is written!

Then execute the following cells.

In [ ]:
entity_id = "PASTE ENTITY ID IN HERE"
tgt_sections = "PASTE SECTION IDS HERE"


if not isinstance(tgt_sections, list):
    tgt_sections = [tgt_sections]

client = Client(project_context=project_context, token_manager=token, environment=env_)
entity = client.get_entity(entity_id=entity_id, entity_type=CellMorphology)

display.display(entity.description)

morphology = CellMorphologyFromID(id_str=entity_id)
neuron_path = "downloaded_morphology.h5"
morphology.write_spiny_neuron_h5(neuron_path, db_client=client)
m = load_morphology_with_spines(neuron_path, load_meshes=True)
print(f"Spine count: {m.spines.spine_count}")

## Mesh
Download source cell mesh

We find the cell surface mesh of the neuron and download it. This is the reference that we proofread against.

In [ ]:
# Load mesh
import pylmesh
subsample_factor = 2

mesh_path = "neuron_mesh.glb"
src_mesh_entity = morphology.source_mesh_entity(db_client=client)
client.download_file(entity_id=src_mesh_entity.id, entity_type=EMCellMesh,
                     asset_id=src_mesh_entity.assets[0], output_path=mesh_path)

mesh_obj = pylmesh.load_mesh(str(mesh_path))
vtx_surf = mesh_obj.vertices[::subsample_factor]
mesh_obj = None
vtx_surf = (numpy.array([[v.x, v.y, v.z] for v in vtx_surf]) * 1E-3).astype(numpy.float32)

sec_points = pandas.concat([
    pandas.DataFrame(sec_.points[:, :3], columns=["x", "y", "z"])
    for sec_ in m.morphology.sections
], axis=0, keys=range(len(m.morphology.sections)), names=["sec_id"]).droplevel(1)

tree = KDTree(sec_points)
_, i = tree.query(vtx_surf)

vtx_surf = pandas.DataFrame(vtx_surf, index=sec_points.index[i])

# Launch 3d visualization

Below, we launch the 3d visualization and error reporting widget.

Note that the code cell is quite long, you might have to do a lot of scrolling.

The instructions for what to do are above. Briefly:
  1. A detected spine is displayed
  2. If it has been wrongly detected and really is no spine, check the box "Is no spine".
  3. Otherwise use the slider to indicate the number of true spines that the detected spine contains, and check all error types that apply. Then press the "Save and next" button
  4. Once all assigned spines have been tackled, the save button disappear. Continue with the rest of the notebook.

**NOTE** There is no "undo" function. If you make a mistake, you have to re-run this cell and start from scratch! BE CAREFUL!

In [ ]:
import numpy as np
import k3d
import os
import h5py
import ipywidgets as widgets

spine_count_lbl = widgets.Label("NaN spines left")

# State: track spine mesh objects for clean removal
spine_meshes = []
screenshot_spine_ids = []
screenshots = []

class ProofreadingState(object):
    def __init__(self, fn):
        self.current_spine = [None]
        self.results = []
        self.remaining_spines = list(m.spines.spine_table.set_index("afferent_section_id").loc[tgt_sections, "spine_id"].to_numpy())
        if os.path.isfile(fn):
            with h5py.File(fn, "r") as h5:
                loc = h5["spine_id"]
                rem = h5["remaining"]
                if (loc.attrs["entity_id"] == entity_id) and (len(tgt_sections) == len(rem.attrs["tgt_sections"])) and (set(tgt_sections) == set(rem.attrs["tgt_sections"])):
                    self.current_spine = [loc[0]]
                    results_df = pandas.read_hdf(fn, "results")
                    _, a = zip(*results_df.iterrows())
                    self.results = [a_.to_dict() for a_ in a]
                    self.remaining_spines = list(h5["remaining"][:])
    
    def to_h5(self, fn):
        with h5py.File(fn, "w") as h5:
            loc = h5.create_dataset("spine_id", data=self.current_spine)
            loc.attrs["entity_id"] = entity_id
            rem = h5.create_dataset("remaining", data=self.remaining_spines)
            rem.attrs["tgt_sections"] = tgt_sections
        results_df = pandas.DataFrame(self.results)
        results_df.to_hdf(fn, key="results")
pr_fn = "proofreading_state.h5"
state = ProofreadingState(pr_fn)

# Create K3D plot
plot = k3d.plot(grid_visible=False, background_color=0xffffff)

# Draw the neuron mesh as point cloud (constant screen-space size)
plot += k3d.points(
    vtx_surf.to_numpy(),
    point_size=0.05,
    color=0x88CCEE,
    opacity=0.5,
    shader='gaussian',
)


def display_next_spine(_ignore):
    """Display a spine (head/neck) and focus the camera on it."""
    global spine_meshes, plot, state

    if len(state.remaining_spines) == 0: return False
    
    spine_id = state.remaining_spines.pop()
    state.current_spine.clear()
    state.current_spine.append(spine_id)
    spine_count_lbl.value = f"{len(state.remaining_spines) + 1} spines left"

    # Remove previous spine meshes
    for obj in spine_meshes:
        try:
            plot -= obj
        except Exception:
            pass
    spine_meshes.clear()

    neck = m.spines.spine_mesh(spine_id, include_head=False)
    head = m.spines.spine_mesh(spine_id, include_neck=False)

    if len(neck.faces) > 0:
        obj = k3d.mesh(
            neck.vertices.astype(np.float32),
            neck.faces.astype(np.uint32),
            color=0x64c864,
            flat_shading=True,
        )
        plot += obj
        spine_meshes.append(obj)

    if len(head.faces) > 0:
        obj = k3d.mesh(
            head.vertices.astype(np.float32),
            head.faces.astype(np.uint32),
            color=0xff6464,
            flat_shading=True,
        )
        plot += obj
        spine_meshes.append(obj)

    # Focus camera on the spine
    full = m.spines.spine_mesh(spine_id)
    center = full.centroid.astype(np.float32)
    radius = float(np.linalg.norm(full.vertices - center, axis=1).max())
    camera_pos = center + np.array([0, 0, 4.0 * radius], dtype=np.float32)
    plot.camera_auto_fit = False
    plot.camera = camera_pos.tolist() + center.tolist() + [0, 1, 0]

    actual_spine_count.value = 1
    is_inaccurate.value = False
    is_bogus.value = False
    no_head.value = False
    miss_head.value = False
    split.value = False
    if is_no_spine.value:
        is_no_spine.value = False
    return True

def save_and_move_on(_ignore):
    global state
    assert len(state.current_spine) == 1
    if is_no_spine.value:
        spine_cnt = 0
    else:
        spine_cnt = actual_spine_count.value
    res = {
        "spine_id": state.current_spine[0],
        "spine_count": spine_cnt,
        "merged": actual_spine_count.value > 1,
        "shifted": is_inaccurate.value,
        "broken": is_bogus.value,
        "false_pos_head": no_head.value,
        "missing_head": miss_head.value,
        "split": split.value
    }
    state.results.append(res)
    state.to_h5(pr_fn)

    if not display_next_spine(_ignore):
        next_button.layout.visibility = "hidden"

def take_screenshot(ignore_):
    global plot
    global current_spine
    screenshot_spine_ids.append(state.current_spine[0])
    if len(plot.screenshot) > 0:
        screenshots.append(plot.screenshot)
    plot.fetch_screenshot()

def toggled(change):
    if change["type"] == 'change':
        if isinstance(change["new"], bool):
            if change["new"]:
                for element in [is_inaccurate, is_bogus, no_head, miss_head, split]:
                    element.value = False
                    element.layout.visibility = "hidden"
                spine_count_row.layout.visibility = "hidden"
                actual_spine_count.value = 1
            else:
                for element in [is_inaccurate, is_bogus, no_head, miss_head, split]:
                    element.layout.visibility = "visible"
                spine_count_row.layout.visibility = "visible"

next_button = widgets.Button(description="Save and next")
next_button.on_click(save_and_move_on)
screenshot_button = widgets.Button(description="Screenshot")
screenshot_button.on_click(take_screenshot)
is_no_spine = widgets.Checkbox(description="Is no spine", value=False)
lbl1 = widgets.Label("Mesh reconstruction errors:")
actual_spine_count = widgets.IntSlider(min=1, max=5, value=1)
spine_count_row = widgets.HBox([
    widgets.Label("Actual spine count"),
    actual_spine_count
])
split = widgets.Checkbox(description="Spine split", value=False)
lbl2 = widgets.Label("Head/neck separation errors:")
is_inaccurate = widgets.Checkbox(description="Separation shifted", value=False)
is_bogus = widgets.Checkbox(description="Separation broken", value=False)
no_head = widgets.Checkbox(description="False head", value=False)
miss_head = widgets.Checkbox(description="Missing head", value=False)
err_row = widgets.HBox([is_inaccurate, is_bogus, no_head, miss_head])

display_next_spine(None)
is_no_spine.observe(toggled)

# Layout and display
controls = widgets.HBox([spine_count_lbl, next_button, screenshot_button])
display.display(widgets.VBox([plot, controls, is_no_spine, lbl1, split, spine_count_row, lbl2, err_row]))


## Display error counts

In [ ]:
screenshots.append(plot.screenshot)
results_df = pandas.DataFrame(state.results)

All results DataFrame

In [ ]:
display.display(results_df)

False positive count

In [ ]:
display.display((results_df["spine_count"] == 0).sum())

Total counts

In [ ]:
display.display(results_df[["spine_count", "merged", "shifted", "broken", "false_pos_head", "missing_head", "split"]].sum())

Distribution of number of mesh reconstruction errors

In [ ]:
display.display(results_df[["split", "merged"]].value_counts())

Distribution of number of head/neck separation errors


In [ ]:
display.display(results_df[["shifted", "broken", "false_pos_head", "missing_head"]].value_counts())

# Display screenshots

In [ ]:
from base64 import b64decode

for shot_loc, shot in zip(screenshot_spine_ids, screenshots):
    print(f"Spine {shot_loc}")
    display.display(display.Image(b64decode(shot)))

## Refresh authentication
Proofreading can take a long time. Here, we refresh the authentication with the platform in case it timed out.

In [ ]:
token = obi_auth.get_token(environment=env_, auth_mode="daf")

Quick reminder of what we proofread

In [ ]:
display.display(entity_id)
display.display(entity.description)
display.display(tgt_sections)

# Register the result

**READ THIS CAREFULLY**
This is the part where your report is sent back to us. To make sure this works properly follow the instructions to the letter.

First: SAVE THE STATE OF THIS NOTEBOOK. You can do this by clicking the save button above of using ctrl+s or command+s.

Then execute the following cell. A widget will be displayed where you enter an informative name for your report, such as "Billy proofreads neuron 10e99744-5dd6-4a0e-8713-ccf666cc259e".

In the description please write the `entity_id` in the first line and a comma-separate list of your assigned `section ids` in the second line.

**THEN PRESS THE REGISTER BUTTON**

Don't forget to press the button. The report is only sent after you press the button!

In [ ]:
from obi_notebook.register_notebook_result import register_result
client = Client(project_context=project_context, token_manager=token, environment=env_)
register_result.register(client)